# RPO

- Arbitrary RAAN, inclination, and phasing
- Angles, then 3 DOF, then 6 DOF

The original step-by-step plan lives in `rpo_legacy.ipynb`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from cesium.constants import DAY_TO_SEC, DEG_TO_RAD, KM_TO_M, M_TO_KM, MIN_TO_SEC
from cesium.guidance import RpoTimeline
from cesium.guidance.coplanar import guid_coaxial_hohmann, guid_phasing_same_orbit
from cesium.guidance.hcw import hcw_two_impulse
from cesium.guidance.noncoplanar import guid_combined_i_raan_change, node_line_finder
from cesium.math.base import unit
from cesium.math.rotations import wrap_to_2pi, wrap_to_pi
from cesium.math.time import jd_to_greg, jd_to_sec_j2000
from cesium.orbital.coes import COES, orbit_sweep
from cesium.plotting import plot_3d, plot_rsw, plot_separation
from cesium.world.constants import J2_MARS, MU_MARS, R_MARS
from numpy.linalg import norm

## Setup
Assume within 5 degrees inclination, raan, 0.05 e, 100km a

In [3]:
# Target orbit (same as arrival.ipynb), MCI, circular
PARKING_ORBIT_ALT_M = 300e3  # m
PARKING_ORBIT_RAAN_DEG = 15
PARKING_ORBIT_I_DEG = 45
PARKING_ORBIT_TA_DEG = 13

TARGET_0 = COES(
    a=R_MARS + PARKING_ORBIT_ALT_M,
    e=0.0,
    i=PARKING_ORBIT_I_DEG * DEG_TO_RAD,
    raan=PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
    aop=0.0,
    ta=PARKING_ORBIT_TA_DEG * DEG_TO_RAD,
)
a_t, e_t, i_t, raan_t, aop_t, ta_t = TARGET_0

dt = 1  # sec

In [4]:
# Chaser = target + seeded uniform dispersion
# SEED = 525600
SEED = 3214359876
# SEED = 13
ANGLE_DISP_DEG = 3.0  # i, raan, aop
A_DISP_M = 100 * KM_TO_M
E_MAX = 0.01  # e >= 0, so draw from [0, E_MAX]
J2 = 0
INITIAL_TA_OFFSET_DEG = 57

# CRAZY
# ANGLE_DISP_DEG = 50.0  # i, raan, aop
# A_DISP_M = 500 * KM_TO_M
J2 = J2_MARS

rng = np.random.default_rng(SEED)
rel_coes = np.array(
    [
        rng.uniform(-A_DISP_M, A_DISP_M),
        rng.uniform(0.0, E_MAX),
        *rng.uniform(-ANGLE_DISP_DEG, ANGLE_DISP_DEG, 3) * DEG_TO_RAD,
        INITIAL_TA_OFFSET_DEG * DEG_TO_RAD,  # ta: phasing left to the RPO problem
    ]
)
CHASER_0 = COES.from_array(TARGET_0.elements() + rel_coes)
a_c, e_c, i_c, raan_c, aop_c, ta_c = CHASER_0

TARGET_0.print("initial target coes")
CHASER_0.print("initial chaser coes")

# mean motion
n_c0 = np.sqrt(MU_MARS / a_c**3)
n_t0 = np.sqrt(MU_MARS / a_t**3)

initial target coes: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 13.000 deg
initial chaser coes: a = 3702.270 km, e = 0.00621, i = 44.231 deg, raan = 17.855 deg, aop = -2.894 deg, ta = 70.000 deg


In [5]:
# Epoch of t = 0 on the timeline. Placeholder: the Lambert arrival reference from
# arrival.ipynb (assumed TDB), until the real RPO start epoch comes from the arrival plan.
MISSION_NAME = "birthday-christmas-earlier"
mission = np.load(f"trajectories/{MISSION_NAME}.mission", allow_pickle=False)
EPOCH0_JD_TDB = float(mission["lambert_arrival_jd"])
EPOCH0_TDB_S = jd_to_sec_j2000(EPOCH0_JD_TDB)

print(
    "RPO start: {:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f} TDB".format(
        *jd_to_greg(EPOCH0_JD_TDB)
    ),
    f"({EPOCH0_TDB_S:.3f} s past J2000)",
)

RPO start: 2035-12-25 00:00:00.000 TDB (1135425600.000 s past J2000)


## Burn timeline
One propagated truth for both spacecraft (`cesium.guidance.RpoTimeline`): every `coast` continues from the last state, every `burn` is applied to it, and each burn is planned from the current (propagated) COES. Times are seconds since `EPOCH0_TDB_S`; `tl.epochs_tdb_s()` and `tl.burn_table()` give absolute epochs.

In [ ]:
K_PHASE = 20
HCW_THRESH = 20 * KM_TO_M

tl = RpoTimeline(
    CHASER_0.to_rv(MU_MARS),
    TARGET_0.to_rv(MU_MARS),
    MU_MARS,
    j2=J2,
    r_cb=R_MARS,
    epoch0_tdb_s=EPOCH0_TDB_S,
    frame="MCI",
)

# Burn 1: coast to the line of nodes, combined i/RAAN change
c, t = tl.coes()
n_hat, d_node, _, _ = node_line_finder(tl.chaser, tl.target)
tl.coast(orbit_sweep(c, d_node, MU_MARS)[1])
c, t = tl.coes()
dv = guid_combined_i_raan_change(c.i, t.i, t.raan - c.raan, norm(tl.chaser[3:]))
tl.burn(dv * unit(c.h(MU_MARS)), "Burn 1 (plane)")

# Burns 2-3: coast to the nearer apsis, Hohmann to the target radius
c, t = tl.coes()
peri = c.ta > np.pi
tl.coast(
    orbit_sweep(c, (2 * np.pi - c.ta) if peri else wrap_to_2pi(np.pi - c.ta), MU_MARS)[
        1
    ]
)
c, t = tl.coes()
_, dt_trans, dv_a, dv_b = guid_coaxial_hohmann(
    c.r_p, c.r_a, t.a, t.a, depart_at_periapsis=peri, mu=MU_MARS
)
tl.burn(dv_a * unit(tl.chaser[3:]), "Burn 2 (Hohmann)")
tl.coast(dt_trans)

# Look at phasing to see what to do
v_hat = unit(tl.chaser[3:])
rv_circ = tl.chaser.copy()
rv_circ[3:] += dv_b * v_hat
c_circ = COES.from_rv(rv_circ, MU_MARS)
_, t = tl.coes()

dv_ph, dt_ph, _ = guid_phasing_same_orbit(
    t.a,
    wrap_to_pi(c_circ.aol - t.aol),
    MU_MARS,
    K_PHASE,
    K_PHASE,
    deg=False,
    r_cb=R_MARS,
)

# One real burn: transfer ellipse -> phasing orbit directly
tl.burn((dv_b + dv_ph / 2) * v_hat, "Burn 3 (circularize + phase entry)")
tl.coast(dt_ph / K_PHASE)
# tl.burn(-dv_ph / 2 * unit(tl.chaser[3:]), "Burn 4 (phase exit)")

to_burn = -dv_ph / 2
# tl.burn(dv_b * unit(tl.chaser[3:]), "Burn 3 (circularize)")

# Burns 4.x: same-orbit phasing
sep = [np.inf]
i = 0
phases_remaining = K_PHASE
while phases_remaining:
    c, t = tl.coes()
    dv_ph, dt_ph, _ = guid_phasing_same_orbit(
        t.a,
        wrap_to_pi(c.aol - t.aol),
        MU_MARS,
        phases_remaining,
        phases_remaining,
        deg=False,
        r_cb=R_MARS,
    )

    actual_burn = dv_ph / 2 + to_burn
    tl.burn(actual_burn * unit(tl.chaser[3:]), f"Burn 4.{i} (phase entry)")
    to_burn = -dv_ph / 2

    # If enough out of wack
    # if (norm(c.a - t.a) / t.a < 1e-3  # 0.1 percent
    #     or norm(c.e - t.e) / t.e < 1e-4): # 0.01 percent
    #     c, t = tl.coes()
    #     peri = c.ta > np.pi
    #     tl.coast(orbit_sweep(c, (2 * np.pi - c.ta) if peri else wrap_to_2pi(np.pi - c.ta), MU_MARS)[1])
    #     c, t = tl.coes()
    #     _, dt_trans, dv_a, dv_b = guid_coaxial_hohmann(c.r_p, c.r_a, t.a, t.a, depart_at_periapsis=peri, mu=MU_MARS)
    #     tl.burn(dv_a * unit(tl.chaser[3:]), f"Burn 4.{i}b (Hohmann)")
    #     tl.coast(dt_trans)
    #     tl.burn(to_burn * unit(tl.chaser[3:]), f"Burn 4.{i}c (Hohmann)")
    #     to_burn = 0

    i += 1
    phases_remaining -= 1

    c, t = tl.coes()
    t_phase = c.period(MU_MARS)
    tl.coast(t_phase)

    sep = norm(tl.chaser[0:3] - tl.target[0:3])
    # print(zsep)
    if sep < HCW_THRESH:
        break


print(f"Exited with {phases_remaining=}")

# Burn 5: same-orbit phasing
tl.burn(to_burn * unit(tl.chaser[3:]), "Burn 5 (Final phase exit)")

# Burns 6-7: same-orbit phasing

c, t = tl.coes()
t_rendezvous = 40 * MIN_TO_SEC
for i in range(3):
    rv_c_rsw = tl.to_rsw()
    dv1_RSW, dv2_RSW = hcw_two_impulse(
        rv_c_rsw[0:3], rv_c_rsw[3:6], t_rendezvous, tl.coes()[1].n(MU_MARS)
    )

    rv_rsw_burn1 = np.concatenate((rv_c_rsw[0:3], rv_c_rsw[3:6] + dv1_RSW))
    rv_intrl_burn1 = tl.from_rsw(rv_rsw_burn1)
    tl.burn(rv_intrl_burn1[3:6] - tl.chaser[3:6], f"HCW {i}A")
    tl.coast(t_rendezvous)

    rv_c_rsw = tl.to_rsw()
    rv_rsw_burn2 = np.concatenate((rv_c_rsw[0:3], rv_c_rsw[3:6] + dv2_RSW))
    rv_intrl_burn2 = tl.from_rsw(rv_rsw_burn2)
    tl.burn(rv_intrl_burn2[3:6] - tl.chaser[3:6], f"HCW {i}B")

tl.coast(1 * DAY_TO_SEC)

# print(tl.burn_table())
print(f"total dv = {tl.dv_total:.3f} m/s")

ts_tl, rv_c_tl, rv_t_tl = tl.arrays()
# fig_sep = plot_separation(ts_tl, rv_c_tl, rv_t_tl, tl.burn_events())
fig_sep_close = plot_separation(
    ts_tl, rv_c_tl, rv_t_tl, tl.burn_events(), top=50, left=2500, units="m"
)  # within 1 km

Exited with phases_remaining=0
total dv = 259.261 m/s


### HCW Targeting

In [13]:
print(tl.burn_table())

  #     t [hr]           JD (TDB)  |dv| [m/s]  name
  0      1.183     2464686.549294     123.328  Burn 1 (plane)
  1      1.478     2464686.561585      13.265  Burn 2 (Hohmann)
  2      2.422     2464686.600896      10.168  Burn 3 (circularize + phase entry)
  3      4.325     2464686.680227       0.263  Burn 4.0 (phase entry)
  4      6.229     2464686.759547       0.171  Burn 4.1 (phase entry)
  5      8.133     2464686.838882       0.180  Burn 4.2 (phase entry)
  6     10.038     2464686.918232       0.191  Burn 4.3 (phase entry)
  7     11.942     2464686.997596       0.203  Burn 4.4 (phase entry)
  8     13.847     2464687.076976       0.216  Burn 4.5 (phase entry)
  9     15.753     2464687.156372       0.232  Burn 4.6 (phase entry)
 10     17.659     2464687.235784       0.249  Burn 4.7 (phase entry)
 11     19.565     2464687.315213       0.270  Burn 4.8 (phase entry)
 12     21.472     2464687.394661       0.294  Burn 4.9 (phase entry)
 13     23.379     2464687.474128       

In [8]:
ts_rsw, rv_rsw = tl.rsw_arrays(progress=True)
burns_tl = tl.burn_events()
plot_rsw(ts_rsw[::5], rv_rsw[::5], burns_tl, show=False)

# Zoom on phasing: everything from burn 3 on
# m = ts_rsw >= tl.burns[2].t_s
# plot_rsw(ts_rsw[m], rv_rsw[m], burns_tl[2:])

100%|██████████| 27973/27973 [00:01<00:00, 25394.66it/s]


In [9]:
# 3D view of the chaser in the target's RSW frame (plot X/Y/Z = R/S/W), target at the origin
# Zoom on HCW: m = ts_rsw >= next(t_b for t_b, name in burns_tl if name.startswith("HCW"))
m = np.ones_like(ts_rsw, dtype=bool)
r_rel_rsw = rv_rsw[m, 0:3]
ts_rel = ts_rsw[m]

axis_len = 0.01 * norm(r_rel_rsw, axis=1).max()  # [m]
rsw_axes = [
    {
        "direction": [1, 0, 0],
        "length": axis_len,
        "name": "R",
        "color": "red",
        "label": True,
    },
    {
        "direction": [0, 1, 0],
        "length": axis_len,
        "name": "S",
        "color": "green",
        "label": True,
    },
    {
        "direction": [0, 0, 1],
        "length": axis_len,
        "name": "W",
        "color": "blue",
        "label": True,
    },
]

burn_points = [
    {"name": "Target", "location": np.zeros(3), "color": "black", "symbol": "x"}
]
for t_b, name in burns_tl:
    if t_b >= ts_rel[0]:
        k = np.argmin(np.abs(ts_rel - t_b))
        burn_points.append(
            {"name": name, "location": r_rel_rsw[k], "color": "orange", "size": 4}
        )
burn_points = []
fig_rsw_3d = plot_3d(
    r_rel_rsw[-10000::5],
    ts_rel[-10000::5],
    labels="Chaser (rel. target)",
    trace_colors="royalblue",
    title="Chaser relative to target (RSW: X = R, Y = S, Z = W)",
    vectors=rsw_axes,
    points=burn_points,
    scale=KM_TO_M,  # inputs in m, displayed in km
    units="km",
)

In [10]:
ts_tl, rv_c_tl, rv_t_tl = tl.arrays()
fig_sep = plot_separation(ts_tl, rv_c_tl, rv_t_tl, tl.burn_events())

## Visualization

In [11]:
vectors = [
    {
        "direction": n_hat,
        "length": 20000,
        "name": "Node line",
        "color": "goldenrod",
        "label": True,
    },
]

fig = plot_3d(
    [rv_t_tl[:, 0:3] * M_TO_KM, rv_c_tl[:, 0:3] * M_TO_KM],
    [ts_tl, ts_tl],
    labels=["Target", "Chaser"],
    trace_colors=["black", "royalblue"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title="Target and chaser, full timeline (MCI)",
    vectors=vectors,
    units="km",
)

# end